# 09 — The propositions, participant round (§4.3)

C02, for the real round. Run it **after the round closes**, and only after the
gate (C03) has passed. It is the same analysis as `06_analysis`, which was
validated on the pilot. The parameters below are fixed by D96–D108; change
them only with a `DECISIONS.md` entry. Every section says "no data yet" rather
than failing while the round is empty.

**Descriptive only (D104).** Counts, proportions and distributions; no
statistical test and no interval. The participant sample is a convenience
sample with no defined population, and turns are nested within participants.
Each proposition is judged against a **criterion fixed before the data**
(`CRITERIA` below, D109), and every assessment cell prints the observed value
next to it.

Sections, in §4.3's order:

1. **Descriptives.** Sessions, request types and shapes, and the two feedback
   axes (Table 12 as two axes, D102).
2. **P2: failures the system recovered from** (D103). Tool calls that errored
   in a turn that still answered, per tool, both rounds side by side, with the
   kind of each error and empty lookups as a secondary row.
3. **P1: coverage.** Requests (D100) and turns whose D24 signature the
   simulated runs produced, both sides segmented by the same rule (D108), where uncovered ones concentrate (shapes7, D99),
   and why each missed.
4. **P3: judge vs feedback.** Judge-score distributions side by side, thumbs
   and accept/reject kept apart. Blocked on Q35.
5. **Usability.** UMUX-LITE, SUS equivalent, curved grade, recommend item.
6. **Per participant.** A shallow overview, to see whether a pooled number is
   carried by one person. Not a result of its own.

The logic lives in `src/analysis.py` (tested in `tests/test_analysis.py`), the
figures in `src/figures.py`.

**Privacy.** This notebook reads participant rows. Outputs are stripped on
commit (`notebooks/study/.gitattributes`). Nothing prints a participant's
words, an id or an email. Figures go to `figures/<run_type>/`, which is
git-ignored; they reach the thesis by being copied there on purpose.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1]))

import matplotlib.pyplot as plt
import pandas as pd

from src import analysis, db, figures, pilot, shapes
from src.execution import _place_ids_of, latest_attempts
from src.signature import SIGNATURE_RULE_VERSION

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)

# --- parameters -------------------------------------------------------------
RUN_TYPE = "study"
SIM_RUN_IDS = [
    "1a872c09-7070-4789-a806-904969f9185d",  # B10 baseline run, adopted (D96)
    "e097f78e-3326-4aea-acaa-0b904d3b0a1c",  # B11 regression run
]
# The criterion each proposition is judged against, as the author fixes it
# before the study round (Q37). None = not yet fixed; the assessment cells then
# print the observed value only.
CRITERIA = {  # D109 (closes Q37); the thresholds live in src/analysis.py
    "P1": "more than half of the participant requests covered (per request vs segments, D108)",
    "P2": "at least 2 of the simulation's 3 highest-error tools among the participant round's 3, "
          "over tools called at least 10 times in each round; not assessed if fewer than 2 of the "
          "simulation's 3 reach 10 participant calls",
    "P3": "thumbs-up replies score at least 1 point higher on average than thumbs-down replies; "
          "not assessed below 10 scored replies in either group",
}
save = figures.Saver(RUN_TYPE, enabled=True)
figures.setup()
pct = "{:.1%}".format

def rate_frame(rates):
    return pd.DataFrame([vars(r) for r in rates], columns=["group", "n", "failures", "rate"]).set_index("group")

def side_by_side(sim_rates, part_rates, order=None):
    """Both rounds per class: n, failures, rate. Rates as percentages."""
    t = rate_frame(sim_rates).join(rate_frame(part_rates), lsuffix=" sim", rsuffix=" part", how="outer")
    if order:
        t = t.reindex(list(order) + sorted(set(t.index) - set(order)))
    return t.style.format({"rate sim": "{:.1%}", "rate part": "{:.1%}"}, na_rep="–") \
        .format({c: "{:.0f}" for c in ("n sim", "failures sim", "n part", "failures part")}, na_rep="–")

def rate_figure(rounds, title, xlabel, name, **kw):
    n = len({r.group for rates in rounds.values() for r in rates})
    fig, ax = plt.subplots(figsize=(6.4, 0.5 * max(n, 1) + 1.2))
    figures.rate_plot(ax, rounds, title, xlabel, **kw)
    save(fig, name); plt.show()

def assess(prop, observed, verdict):
    """Print what was observed, the criterion fixed for it (D109) and the verdict."""
    print(f"{prop} — observed: {observed}")
    print(f"{prop} — criterion: {CRITERIA[prop]}")
    print(f"{prop} — verdict: {verdict.outcome.upper()} ({verdict.reason})")

client = db.admin()
data = pilot.load(client, RUN_TYPE)
names = data.pseudonyms()
turns = list(data.suggestions)
replies = [t["reply_message_id"] for t in turns if t.get("reply_message_id")]
refs = {m["id"]: _place_ids_of(m.get("metadata"))
        for m in (client.table("chat_messages").select("id, metadata").in_("id", replies).execute().data
                  if replies else [])}
requests = [seg for run in analysis.runs_of(turns) for seg in analysis.segments(run, refs)]

sim_executions, superseded = analysis.simulated_executions(client, SIM_RUN_IDS)
sim_requests = analysis.simulated_requests(sim_executions)  # D108: segmented like a participant run
sim_turns = [t for ex in sim_executions for t in ex]

print(f"participant ({RUN_TYPE}): {len(data.sessions)} sessions, {len(turns)} turns, {len(requests)} requests")
print(f"simulation: {len(SIM_RUN_IDS)} runs, {len(sim_executions)} executions ({len(sim_requests)} segments), "
      f"{len(sim_turns)} turns "
      f"({superseded} superseded attempts left out, D53)")
print(f"rules: signature {SIGNATURE_RULE_VERSION}, shape {shapes.REQUEST_SHAPE_RULE_VERSION}, "
      f"analysis {analysis.ANALYSIS_VERSION}")

## 1. Descriptives

Sessions are pseudonymous (`S1, …` in consent order, `D1, …` declined).
Requests are Tonic runs split under D100.

In [ ]:
sessions = pd.DataFrame(pilot.session_table(data))
if len(sessions):
    consented = sessions[sessions.status == "consented"]
    print(f"consented sessions: {len(consented)}; turns per session: median {consented.turns.median()}, "
          f"range {consented.turns.min()}–{consented.turns.max()}")
    display(sessions)
else:
    print("no sessions yet")

In [ ]:
dist = pd.DataFrame({
    "participant turns": pd.Series([t.get("request_type") or "(none)" for t in turns], dtype=object).value_counts(),
    "simulated turns": pd.Series([t.get("request_type") or "(none)" for t in sim_turns], dtype=object).value_counts(),
}).fillna(0).astype(int)
shape_dist = pd.DataFrame({
    "participant requests": pd.Series([analysis.interaction_shape(r) for r in requests], dtype=object).value_counts(),
    "participant turns": pd.Series([analysis.turn_shape(t) for t in turns], dtype=object).value_counts(),
    "simulated turns": pd.Series([analysis.turn_shape(t) for t in sim_turns], dtype=object).value_counts(),
}).fillna(0).astype(int)
display(dist)
shape_dist

In [ ]:
# Figure 1 — shape distribution, share of turns per round (shapes7, D99).
def shares(col):
    total = shape_dist[col].sum()
    return (shape_dist[col] / total).to_dict() if total else {}

fig, ax = plt.subplots(figsize=(6.4, 0.42 * max(len(shape_dist), 1) + 1))
figures.paired_shares(ax, {"simulation": shares("simulated turns"), "participant": shares("participant turns")},
                      f"Request shapes (n = {len(sim_turns)} simulated, {len(turns)} participant turns)",
                      "share of agent turns")
save(fig, "fig1_shapes"); plt.show()

**Table 12 as two axes (D102).** The structural verdict exists only in
simulation (D73) and `user_action` only for participants, so they are reported
side by side, not crossed. §4.3 (1): if `ignored` dominates, it is said here
before anything is interpreted.

In [ ]:
axes = analysis.participant_axes(turns, data.feedback)
ua = pd.Series(axes["user_action"], dtype=int).sort_values(ascending=False)
exec_rows = []
for rid in SIM_RUN_IDS:
    rows = client.table("study_case_executions").select("*").eq("run_id", rid).execute().data
    exec_rows += latest_attempts(rows)[0]
structural = pd.Series([r.get("structural_outcome") or "(not checked)" for r in exec_rows], dtype=object).value_counts()

print("participant — user_action per turn (n = %d):" % ua.sum()); print(ua.to_string() if len(ua) else "  (none)")
if len(ua) and ua.get("ignored", 0) > ua.sum() / 2:
    print("\n>>> ignored dominates: most suggestions got no reaction; read the agreement measures with that in mind.")
print("\nparticipant — thumbs (n = %d):" % sum(axes["thumbs"].values()))
print(pd.Series(axes["thumbs"], dtype=int).to_string() if axes["thumbs"] else "  (none)")
print("\nsimulation — structural outcome per execution (n = %d):" % structural.sum()); print(structural.to_string())

## 2. P2: failures the system recovered from, per tool

**The measure (D103).** A failure is a **tool call that came back an error in
a turn that still produced a response**: the agent hit a problem and answered
anyway. It is counted per tool call and compared **per tool**, both rounds
side by side; request type and shape are secondary views. A turn that failed
outright (`error_code` set, no response) is counted separately. Simulated turns
come from the latest attempt of each execution (D53). The ordering is read off
the table (D104); `n` is shown with every rate, because a rate over a handful of
calls says little.

Steps: **(a)** the headline per round, **(b)** every tool, **(b2)** empty
lookups as a secondary row, **(c)** what kind of error it was, **(d)** request
type and shape, **(e)** the assessment.

In [ ]:
# (a) The headline per round.
def headline(turns_):
    s = analysis.error_summary(turns_)
    done = s["turns"] - s["failed without a response"]
    fmt = lambda k, n, p: f"{k}/{n} = {pct(p)}" if n else "–"
    return {"agent turns": s["turns"], "failed without a response": s["failed without a response"],
            "turns with a recovered tool error": fmt(s["turns with a recovered tool error"], done,
                                                     s["share of answered turns"]),
            "tool calls": s["tool calls"],
            "tool calls that errored": fmt(s["errored tool calls"], s["tool calls"], s["share of calls"])}

pd.DataFrame({"simulation": headline(sim_turns), "participant": headline(turns)})

In [ ]:
# (b) Every tool either round called: calls, errored calls, rate; sorted by the simulated rate.
sim_tool, part_tool = analysis.recovered_error_rates(sim_turns), analysis.recovered_error_rates(turns)
tool_order = [r.group for r in sorted(sim_tool, key=lambda r: (-r.rate, r.group))]
side_by_side(sim_tool, part_tool, order=tool_order)

In [ ]:
# Figure 2 — errored share of calls per tool, both rounds, every tool called.
rate_figure({"simulation": sim_tool, "participant": part_tool},
            "Tool calls that came back an error, in turns that still answered",
            "share of the tool's calls", "fig2_tool_errors", order=tool_order, missing="not called")

**(b2) Secondary row: empty lookups.** A lookup that worked and found
nothing (`result_found = false`): a miss, not a failure. Only tools that
report a `found` flag count, and only their error-free calls.

In [ ]:
sim_miss, part_miss = analysis.empty_lookup_rates(sim_turns), analysis.empty_lookup_rates(turns)
side_by_side(sim_miss, part_miss) if (sim_miss or part_miss) else "no tool with a found flag was called in either round"

**(c) What went wrong.** The kind of each error, read off the tool's error
message by pattern (`analysis.ERROR_KINDS`). The message itself is never shown,
because it can echo what a participant typed. A message no pattern knows is
counted as `other`.

In [ ]:
kinds = pd.DataFrame({
    "simulation": pd.Series(analysis.error_kinds(sim_turns), dtype=int),
    "participant": pd.Series(analysis.error_kinds(turns), dtype=int),
}).fillna(0).astype(int)
if len(kinds):
    kinds.index = kinds.index.set_names(["tool", "kind"])
    display(kinds.sort_index())
else:
    print("no tool call errored in either round")

**(d) Secondary views.** The same measure per turn: the share of turns of
each request type (D09) and each shape (shapes7, D99) with at least one
recovered tool error.

In [ ]:
for key, label in (("request_type", "request type"), ("shape", "shape (shapes7)")):
    print(label)
    display(side_by_side(analysis.recovered_error_rates(sim_turns, key=key),
                         analysis.recovered_error_rates(turns, key=key)))

In [ ]:
# (e) Assessment. The tools ordered by their recovered-error rate in each round,
# with the calls behind each rate.
def ranked(rates):
    return ", ".join(f"{r.group} {r.failures}/{r.n}" for r in sorted(rates, key=lambda r: (-r.rate, r.group))
                     if r.failures) or "no tool errored"

assess("P2", f"simulation: {ranked(sim_tool)} | participant: {ranked(part_tool)}",
       analysis.p2_verdict(sim_tool, part_tool))

## 3. P1: coverage

A participant request is **covered** when a simulated unit of the runs above
has the same D24 signature: tool sequence with consecutive repeats collapsed,
request type, bucketed calls per tool and turns, and four parameter flags.

**Both sides are segmented by the same rule (D108).** Participant runs are
split into requests under D100, and so is every simulated execution. D100
splits some multi-step executions (search → draft), and a participant doing
exactly the same would be split the same way; matching against whole
executions would put D100's split error on the participant side only. The
headline is **per request against simulated segments**; coverage against
whole executions is reported beside it (the suite's own unit, one case run),
then per turn, and per participant (in section 6). §4.2.3: the uncovered interactions
are of more interest than the proportion.

In [ ]:
cov_requests = analysis.coverage(requests, sim_requests, "request")    # headline (D108)
cov_whole = analysis.coverage(requests, sim_executions, "request")      # against whole executions
cov_turns = analysis.coverage([[t] for t in turns], sim_executions, "turn")
summary = pd.DataFrame({"per request, vs segments (headline, D108)": analysis.coverage_summary(cov_requests),
                        "per request, vs whole executions": analysis.coverage_summary(cov_whole),
                        "per turn": analysis.coverage_summary(cov_turns)}).T
display(summary.style.format({"share covered": "{:.1%}"}, na_rep="–"))

unc = pd.DataFrame({
    "uncovered requests": pd.Series(analysis.uncovered_by(cov_requests), dtype=int),
    "all requests": pd.Series([r.shape for r in cov_requests], dtype=object).value_counts(),
}).fillna(0).astype(int).sort_values("uncovered requests", ascending=False)
unc if len(unc) else "no participant requests yet"

**Why each uncovered request missed.** Every uncovered request against the
closest signature the simulated runs produced, and the parts in which they
differ. One part off, especially `turns` or `calls per tool`, is a near miss:
the suite does this, but not in exactly this way, and D24's matching
tolerance counts it as uncovered. Several parts off, or a `tool sequence`
difference, is a shape the suite does not produce. This is a diagnostic only:
it changes nothing about what is covered. Only tool names and labels are shown,
never request text.

In [ ]:
diagnosis = pd.DataFrame(analysis.uncovered_diagnosis(requests, sim_requests))
if len(diagnosis):
    near = (diagnosis["parts off"] == 1).sum()
    print(f"{len(diagnosis)} uncovered requests: {near} one part off (near miss), "
          f"{len(diagnosis) - near} further off")
    display(diagnosis)
    display(pd.Series([p for d in diagnosis["differs in"] for p in d.split(", ")])
            .value_counts().rename("uncovered requests differing in this part").to_frame())
else:
    print("no uncovered participant request" + ("" if requests else " (no requests yet)"))

In [ ]:
# Figure 3 — covered and uncovered requests per shape.
frame = pd.DataFrame([vars(r) for r in cov_requests])
fig, ax = plt.subplots(figsize=(6.4, 0.42 * max(frame["shape"].nunique() if len(frame) else 1, 1) + 1))
if len(frame):
    counts = frame.groupby(["shape", "covered"]).size().unstack(fill_value=0).reindex(columns=[True, False], fill_value=0)
    counts = counts.loc[counts.sum(axis=1).sort_values().index]
    ax.barh(counts.index, counts[True], color=figures.SERIES, label="covered by the suite", height=0.6)
    ax.barh(counts.index, counts[False], left=counts[True] + 0.04, color="#c3c2b7", label="uncovered",
            height=0.6, hatch="//", edgecolor="white", linewidth=0)
    for i, (c, u) in enumerate(zip(counts[True], counts[False])):
        ax.annotate(f"{c}/{c + u}", (c + u, i), xytext=(6, 0), textcoords="offset points", va="center",
                    fontsize=8, color=figures.MUTED)
    ax.set_xlabel("participant requests; both sides segmented by D100 (D108)")
    ax.grid(axis="x", color=figures.GRID, linewidth=0.8); ax.set_axisbelow(True)
    ax.legend(frameon=False, loc="lower right")
    s = analysis.coverage_summary(cov_requests)
    ax.set_title(f"Coverage: {s['covered']}/{s['n']} requests ({s['share covered']:.0%}) match a simulated signature")
else:
    ax.set_title("Coverage"); figures.empty(ax)
save(fig, "fig3_coverage"); plt.show()

In [ ]:
# Assessment.
s, w = analysis.coverage_summary(cov_requests), analysis.coverage_summary(cov_whole)
assess("P1", f"{s['covered']} of {s['n']} requests covered ({pct(s['share covered'])}) against simulated segments; "
             f"{w['covered']} of {w['n']} ({pct(w['share covered'])}) against whole executions"
       if s["n"] else "no requests yet", analysis.p1_verdict(cov_requests))

## 4. P3: judge score vs user feedback

The judge scores each participant reply blind to what the member did. The
scores are shown side by side for replies that were received positively and
negatively, for thumbs and for accept/reject separately, never pooled.
`ignored` and unset actions belong to neither group. No test (D104).
**Only thumbs are assessed (D109):** a reply usually holds several suggestions
and a member may act differently on each, so accept/reject does not judge the
reply as a whole. It is still shown.

**Blocked on Q35.** No judge scores exist for participant replies yet, and the
judge cannot run on them as built: its rubrics are per test case, not per
request type; it prepends the fixture trip, not the member's trip; and
`study_scores` is keyed to a simulated execution. `PARTICIPANT_SCORES` below
stays empty until Q35 is ruled.

In [ ]:
PARTICIPANT_SCORES: dict[str, float] = {}  # turn id -> judge score; see Q35

groups = analysis.feedback_groups(turns, data.feedback, PARTICIPANT_SCORES)
print(f"{len(PARTICIPANT_SCORES)} participant replies scored")
pd.DataFrame(analysis.feedback_scores(groups)).set_index(["signal", "group"])

In [ ]:
# Figure 4 — judge score by feedback group, one panel per signal.
if PARTICIPANT_SCORES:
    fig, axs = plt.subplots(1, 2, figsize=(6.4, 3.2), sharey=True)
    for ax, (signal, g) in zip(axs, groups.items()):
        for x, key in enumerate(("positive", "negative")):
            vals = g[key]
            ax.scatter([x + (k % 5 - 2) * 0.04 for k in range(len(vals))], vals, s=36,
                       color=figures.ROUND_COLOR["participant"], edgecolor="white", linewidth=1.5, zorder=3)
            if vals:
                ax.plot([x - 0.25, x + 0.25], [pd.Series(vals).median()] * 2, color=figures.INK, linewidth=2)
        ax.set_xticks([0, 1], [f"positive\n(n = {len(g['positive'])})", f"negative\n(n = {len(g['negative'])})"])
        ax.set_title(signal); ax.set_ylim(0.5, 10.5)
    axs[0].set_ylabel("judge score (1–10); bar = median")
    save(fig, "fig4_judge_vs_feedback"); plt.show()
else:
    print("no participant judge scores yet — figure 4 is drawn once there are (Q35)")

In [ ]:
# Assessment.
thumbs = {r["group"]: r for r in analysis.feedback_scores(groups) if r["signal"] == "thumbs"}
assess("P3", ", ".join(f"thumbs {g}: mean {r['mean']}, n = {r['n']}" for g, r in thumbs.items())
       if PARTICIPANT_SCORES else "no judge scores yet (Q35)",
       analysis.p3_verdict(groups["thumbs"]))

## 5. Usability

UMUX-LITE on 0–100 (D18), then the **UMUX-LITE → SUS conversion** (Lewis,
Utesch & Maher, 2013): SUS-equivalent ≈ 0.65 × UMUX-LITE + 22.9, an estimate
of the SUS score rather than a measured one. It is graded on the Sauro–Lewis
curved grading scale. The recommend item is on 1–7.

In [ ]:
usab = pd.DataFrame(analysis.usability(data.post_task))
if len(usab):
    mean_sus = usab.sus_equivalent.mean()
    print(f"n = {len(usab)} submitted surveys; mean UMUX-LITE {usab.umux_lite.mean():.1f}, "
          f"SUS-equivalent {mean_sus:.1f} → grade {analysis.curved_grade(mean_sus)}; "
          f"recommend median {usab.recommend.median()}")
    display(usab.describe().loc[["count", "mean", "std", "min", "50%", "max"]].round(1))
    display(usab.grade.value_counts().rename("surveys").to_frame())
else:
    print("no submitted post-task survey yet")

In [ ]:
# Figure 5 — SUS-equivalent scores against the curved grading scale.
fig, ax = plt.subplots(figsize=(6.4, 1.8))
figures.grade_band(ax, list(usab.sus_equivalent) if len(usab) else [], analysis.CURVED_GRADES,
                   "Usability", "SUS-equivalent score (from UMUX-LITE)", color=figures.ROUND_COLOR["participant"])
save(fig, "fig5_usability"); plt.show()

## 6. Per participant (overview)

One row per consented participant, pseudonymous. It shows whether a pooled
number is carried by one person, or whether participants look alike. It is a
consistency check, not a result of its own: nothing here is assessed, and
with few participants a row can identify someone, so it stays out of the thesis
unless it shows something the pooled numbers hide.

In [ ]:
overview = pd.DataFrame(analysis.participant_overview(
    data.sessions, names, requests, [r.covered for r in cov_requests], data.feedback, data.post_task))
if len(overview):
    with_uncovered = int((overview["requests uncovered"] > 0).sum())
    print(f"participants with at least one uncovered request: {with_uncovered} of {len(overview)}")
    display(overview.set_index("participant"))
else:
    print("no consented participant yet")

In [ ]:
print(f"figures written to {save.dir.resolve()}")